In [ ]:
import pandas as pd
import numpy as np
from iteround import saferound
import random
import re
from rdkit import Chem, RDLogger, rdBase
from rdkit.Chem import AllChem, Descriptors, Draw
from tqdm import tqdm
from tqdm.notebook import tqdm

In [ ]:
tqdm.pandas()

## Determine Mol Distribution & Data Pre-Processing

In [ ]:
polymer_data = pd.read_excel('data/shoshanas_synthesized_polymers.xlsx')
polymer_data

In [ ]:
monomer_data = pd.read_csv('../monomer_data/monomers.csv')
monomer_data['mon_class'] = monomer_data['mon_class'].map(lambda x: str(x).lower())
monomer_data = monomer_data[monomer_data['mon_class'].isin(["cationic", "hydrophobic", "hydrophilic"])]

In [ ]:
cols = ['mon_abb', 'SMILES', 'min_mon_wt', 'max_mon_wt', 'increments', 'molar_mass']
mon_df = monomer_data[cols].rename(columns={'min_mon_wt': 'min', 'max_mon_wt': 'max', 'increments': 'increment'})
monomer_properties = mon_df.set_index('mon_abb').T.to_dict('dict')
monomer_properties

In [ ]:
monomers = polymer_data[polymer_data.columns[polymer_data.columns.str.contains('monomer')].tolist()].values
wt = polymer_data[polymer_data.columns[polymer_data.columns.str.contains('wt_%')].tolist()].values
smiles = np.vectorize(lambda x: monomer_properties[x]['SMILES'])(monomers)
molar_masses = np.vectorize(lambda x: monomer_properties[x]['molar_mass'])(monomers)

mol = wt / molar_masses
mol = mol / mol.sum(axis=1).reshape(len(mol.sum(axis=1)),1)

In [ ]:
new_data = pd.DataFrame({'monomers': monomers.tolist(), 'mon_SMILES': smiles.tolist(), 'wt_%': wt.tolist(), 'mol_distribution': mol.tolist()})

ID_col = pd.DataFrame(polymer_data.index.tolist(), columns=["ID"])
# ID_col = ID_col.map(lambda x: 'ID' + str(x))

cols = ['DP', 'MIC_ecoli', 'MIC_saureus', 'HC50']
polymer_data = pd.concat([ID_col, new_data, polymer_data[cols]], axis=1)
polymer_data['immunogenic'] = polymer_data['MIC_ecoli'].map(lambda x: 'No' if x == '>512' else 'Yes')
# polymer_data.index.name = 'ID'

In [ ]:
polymer_data['immunogenic'].value_counts()

In [ ]:
polymer_data

## Sampling Strategies

In [ ]:
class sample(object):
    def __init__(self, polymer_data, sampling_method, n, batch = False):
        self._IDs = polymer_data['ID']
        self._monomers = dict(zip(self._IDs, polymer_data['monomers']))
        self._dist = dict(zip(self._IDs, polymer_data['mol_distribution']))
        self._DP = dict(zip(self._IDs, polymer_data['DP']))
        self._n = n
        self._batch = batch
        self._sampling_method = sampling_method

        self._dict = dict.fromkeys(polymer_data.index.tolist())

        for ID in self._dict.keys():
            monomer_dist = {self._monomers[ID][i]: self._dist[ID][i] for i in range(len(self._monomers[ID]))}
            monomer_num = {self._monomers[ID][i]: self._dist[ID][i]*self._DP[ID] for i in range(len(self._monomers[ID]))}
            monomer_num = saferound(monomer_num, 0)

            self._dict[ID] = {'dist': monomer_dist, 'mon_num': monomer_num, 'DP': self._DP[ID]}

    def determine_monomer(self, U, cum_dist):
        for i, partition in enumerate(cum_dist):
            if U < partition:
                return i

    def uniform_sample(self, ID):
        polymer = []
        monomers = list(self._dict[ID]['dist'].keys())
        cum_dist = np.cumsum(list(self._dict[ID]['dist'].values()))
        for i in range(self._DP[ID]):
            U = np.random.uniform(0, 1)
            polymer.append(monomers[self.determine_monomer(U, cum_dist)])

        polymer = ''.join(polymer)
        
        return polymer

    def sample_wo_replacement(self, ID):

        dist = self._dict[ID]['mon_num'].copy()

        if self._batch == True:
            dist = {key: val*self._n for key, val in dist.items()}
        
        all_monomers = []
        for key, value in dist.items():
            all_monomers.extend([key] * int(value))
    
        # Shuffle the list to ensure randomness
        random.shuffle(all_monomers)
        
        # Select strings randomly until all values are zero
        polymer = []
        while all_monomers:
            pick = all_monomers.pop()
            polymer.append(pick)
            dist[pick] -= 1
    
        polymer = ''.join(polymer)
        
        return polymer
    
    def generate_n_samples(self, ID):
        
        samples = set()

        if self._sampling_method == 'wo_replacement' and self._batch == True:
            sample = self.sample_wo_replacement(ID)
            sample_split = re.findall('[A-Z][^A-Z]*', sample)
            x = int(len(sample_split)/self._n)
            samples.update([''.join(sample_split[i:i+x]) for i in range(0, len(sample_split), x)])
        else:
            while len(samples) < self._n:
                if self._sampling_method == 'uniform':
                    samples.add(self.uniform_sample(ID))
                if self._sampling_method == 'wo_replacement' and self._batch == False:
                    samples.add(self.sample_wo_replacement(ID))

        return list(samples)
    
    def main(self):

        samples_dict = {}
        
        for ID in self._dict.keys():
            samples_dict[ID] = self.generate_n_samples(ID)
        
        return samples_dict

In [ ]:
# s = sample(polymer_data, 'wo_replacement', 3, batch = True)
# s = sample(polymer_data, 'wo_replacement', 4, batch = False)
# s = sample(polymer_data, 'uniform', 5)
# a = s.generate_n_samples(16)
# for b in a:
#     print(b)
#     print(len(re.findall('[A-Z][^A-Z]*', b)))
# s.main()

## Generate Samples

In [ ]:
s = sample(polymer_data, 'uniform', 200, batch = False)
data = s.main()

In [ ]:
def get_polymer_weights(polymer):
 
    freq = {}
    for mon in polymer:
        if (mon in freq):
            freq[mon] += monomer_properties[mon]['molar_mass']
        else:
            freq[mon] = monomer_properties[mon]['molar_mass']

    return freq

def get_seq_dist(polymer):
    wts = get_polymer_weights(polymer)
    total_wts = sum(list(wts.values()))
    wts = {key: val / total_wts for key, val in wts.items()}

    return wts

In [ ]:
samples_df = pd.concat([pd.DataFrame({'ID': k, 'sequence': v}) for k, v in data.items()])

samples_df['seq_mol_dist'] = samples_df['sequence'].map(lambda x: get_seq_dist(re.findall('[A-Z][^A-Z]*', x)))
cols = ['ID', 'monomers', 'mon_SMILES', 'wt_%', 'mol_distribution', 'DP', 'MIC_ecoli', 'MIC_saureus', 'HC50', 'immunogenic']

samples_df = pd.merge(samples_df, polymer_data[cols], on='ID', how='left')

samples_df['seq_mol_dist'] = samples_df.apply(lambda row: 
                                   [row['seq_mol_dist'].get(mon, 0) for mon in row['monomers'] if mon in row['seq_mol_dist']], axis=1)

In [ ]:
samples_df

In [ ]:
# samples_df.to_csv("uniform.csv")

## Polymerization:

In [ ]:
class polymerize(object):
    def __init__(self, sample, DP, monomer_properties):
        self._sample = sample # string of monomers
        self._DP = DP

        sample_split = re.findall('[A-Z][^A-Z]*', self._sample)
        sample_smiles = [monomer_properties[mon]['SMILES'] for mon in sample_split]
        self._sample_mol = [Chem.MolFromSmiles(reactant) for reactant in sample_smiles]

    def reinitialize_polymer(self, product):
        if len(product) > 1:
            return "Too many products"
        else:
            product_smiles = Chem.MolToSmiles(product[0][0])
            reinit_product = Chem.MolFromSmiles(product_smiles)
            return reinit_product
        
    def initialize_reaction(self):
        rxn1 = AllChem.ReactionFromSmarts(
            "[C:0]=[CH2:1].[CH2:2]=[C:3]>>[Kr][C:0][C:1][C:3][C:2][Xe]"
        )

        A = self._sample_mol.pop(0)
        B = self._sample_mol.pop(0)
        product = rxn1.RunReactants((A, B))
        self.polymer = self.reinitialize_polymer(product)

        return product

    def propagate_reaction(self):

        rxn2 = AllChem.ReactionFromSmarts(
            "[C:0][C:1][C:2][C:3][Xe].[C:4]=[CH2:5]>>[C:0][C:1][C:2][C:3][C:4][C:5][Xe]"
        )

        A = self.polymer
        B = self._sample_mol.pop(0)
        product = rxn2.RunReactants((A, B))
        self.polymer = self.reinitialize_polymer(product)

        return product

    def terminate_reaction(self):

        if self._DP > 2:
            # terminate & remove Xe
            rxn3 = AllChem.ReactionFromSmarts(
                "[C:0][C:1][C:2][C:3][Xe].[C:4]=[CH2:5]>>[C:0][C:1][C:2][C:3][C:4][C:5]"
            )
    
            A = self.polymer
            B = self._sample_mol.pop(0)
    
            product = rxn3.RunReactants((A, B))
            self.polymer = self.reinitialize_polymer(product)
    
            # (removes Kr)
            rxn4 = AllChem.ReactionFromSmarts(
                "[C:0][C:1][C:2][C:3][Kr]>>[C:0][C:1][C:2][C:3]" 
            )
    
            A = self.polymer
            product = rxn4.RunReactants((A,))
            self.polymer = product[0][0]
        else:
            rxn3 = AllChem.ReactionFromSmarts(
                "[C:0]=[CH2:1].[CH2:2]=[C:3]>>[C:0][C:1][C:3]=[C:2]"
            )

            A = self._sample_mol.pop(0)
            B = self._sample_mol.pop(0)
            product = rxn3.RunReactants((A,B))
            self.polymer = product[0][0]

    def run_reaction(self):

        if self._DP > 2:
            self.initialize_reaction()

        if self._DP > 3:
            for i in range(self._DP - 3):
                self.propagate_reaction()
                
        self.terminate_reaction()

    def get_smiles(self):
        return Chem.MolToSmiles(self.polymer)

    def draw_diagram(self):
        return Draw.MolToImage(self.polymer, size=(500,500))        

In [ ]:
# sample = "OlamMoTmaTma"
# polymer = polymerize(sample, 4, monomer_properties)
# polymer.run_reaction()
# polymer.get_smiles()
# polymer.draw_diagram()

In [ ]:
samples_df.loc[:, 'poly_SMILES'] = samples_df.progress_apply(lambda row: polymerize(row['sequence'], row['DP'], monomer_properties), axis=1)

In [ ]:
samples_df['poly_SMILES'].progress_apply(lambda x: x.run_reaction())

In [ ]:
samples_df.loc[:, 'poly_SMILES'] = samples_df['poly_SMILES'].progress_apply(lambda x: x.get_smiles())

In [ ]:
samples_df

In [ ]:
samples_df.to_csv("wo_replacement_batch_True.csv")